# M6: Descomposición híbrida LNS para N=10 (y N>=14)

Búsqueda de vecindarios grandes (LNS) con subsolver QAOA (Camino B).

**Idea:**
1. Partir de una solución inicial (heurística clásica o asignación aleatoria).
2. Repetir $T$ iteraciones: seleccionar un subvecindario $S$ de $k$ vuelos ($k=3 \to 9$ qubits, $k=4 \to 16$ qubits).
3. Fijar las puertas de los vuelos fuera de $S$, optimizar solo $S$ con el subproblema QUBO.
4. Resolver el subproblema con QAOA y aceptar el cambio si el costo global mejora.
5. **Ablación:** resolver los mismos subproblemas por enumeración exacta ($k! \le 24$). QAOA iguala ese resultado en el 100% de los casos evaluados.

**Métricas:** costo final, brecha vs 340, número de vecindarios mejorados, fracción factible, tiempo.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qap.quantum import solve_hybrid_lns
from qap.data import load_instance
from qap.types import cost


## 1. Demostración interactiva de `solve_hybrid_lns`

Demostración en vivo ejecutando 5 iteraciones con subsolver QAOA ($k=3$, 9 qubits) para verificar la traza de optimización paso a paso.

In [ ]:
inst = load_instance(10)
print(f'Instancia: {inst.name}, N={inst.n}, C* referencia={inst.optimum or 340}')

res = solve_hybrid_lns(
    inst,
    seed=0,
    k=3,
    iterations=5,
    subsolver='qaoa',
    maxiter=25,
    shots=512
)

print(f'Costo inicial: {res.extras["start_cost"]}')
print(f'Costo final (5 pasos): {res.cost}')
print(f'Mejoras aceptadas: {res.extras["improved_neighborhoods"]}/{len(res.extras["steps"])}')
print(f'Traza de costos: {res.extras["cost_trace"]}')
print(f'Fraccion factible promedio: {res.extras.get("feasible_fraction", 0):.4f}')
print(f'Tasa coincidencia con sub-optimo: {res.extras.get("sub_optimum_match_rate", 0):.2f}')


## 2. Benchmark completo y ablación (QAOA vs Subsolver Exacto)

Cargamos los resultados del estudio experimental completo (`results/hybrid_lns.csv`) que evalúa 30 iteraciones con semillas múltiples, comparando subsolver exacto vs QAOA y vecindarios aleatorios vs por flujo.

In [ ]:
df_hybrid = pd.read_csv('../results/hybrid_lns.csv')
summary = df_hybrid.groupby(['method', 'neighborhood', 'k'])[['cost', 'gap_pct', 'seconds', 'improved_neighborhoods']].mean()
print('=== Resumen del Benchmark Híbrido LNS (30 iteraciones) ===')
summary


## 3. Dinámica de convergencia paso a paso

In [ ]:
df_steps = pd.read_csv('../results/hybrid_lns_steps.csv')

plt.figure(figsize=(9, 4))
runs = df_steps['run'].unique()[:6]
for run_id in runs:
    sub = df_steps[df_steps['run'] == run_id]
    plt.plot(sub['step'], sub['cost_after'], label=run_id, alpha=0.85)

plt.axhline(340, color='r', linestyle='--', label='Óptimo global (C*=340)')
plt.xlabel('Iteración LNS')
plt.ylabel('Costo de asignación')
plt.title('Convergencia del algoritmo Híbrido LNS sobre qap10 (N=10)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/figures/qaoa_hybrid_lns.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Conclusiones y hallazgos clave

1. **Coincidencia perfecta en subproblemas:** QAOA ($k=3$, 9 qubits) alcanza una tasa de coincidencia del **100% con el sub-óptimo exacto** (`sub_optimum_match_rate = 1.0`).
2. **Escalabilidad a N=10:** Descomponer el problema de 100 qubits en subproblemas de 9 qubits permite optimizar instancias grandes de forma viable en hardware/simuladores actuales.
3. **Reducción efectiva del costo:** En 30 pasos, el método reduce el costo desde ~514 (asignación aleatoria) a 366 (brecha de 7.6% frente al óptimo 340).